# 03 — Model Building

This notebook defines the `SentimentLSTM` architecture, explains every design decision, and smoke-tests the model before any training occurs.

**Prerequisites:** `02_preprocessing.ipynb` must have been run first so that `data/vocab.pkl` exists.

**Notebook outline:**
1. Load Vocabulary
2. Define Model
3. Model Summary
4. Smoke Test
5. Two-Layer Variant (Experiment Preview)

In [ ]:
import sys, os

# ── Colab vs. local ortam tespiti ────────────────────────────────────────
if os.path.exists("/content"):
    repo_root = "/content/IMDb_Sentiment_Analysis"
    if not os.path.exists(repo_root):
        os.system("git clone https://github.com/azrakarakaya1/IMDb_Sentiment_Analysis.git " + repo_root)
    os.chdir(repo_root)
else:
    os.chdir(os.path.abspath(".."))

sys.path.insert(0, os.path.abspath("."))
print(f"Working directory: {os.getcwd()}")
import torch
import pickle

## Section 1 — Load Vocabulary

Before we can instantiate the model we need to know the vocabulary size. The `SentimentLSTM` embedding layer is a lookup table with one row per vocabulary token — so `vocab_size` directly determines the number of rows (and therefore the number of parameters) in that table.

We load the `Vocabulary` object that was serialised by `02_preprocessing.ipynb`. If the file is missing, run that notebook first.

In [ ]:
from src.preprocessing import Vocabulary

# Load the vocabulary built from the training split in 02_preprocessing.ipynb.
# Vocabulary.load() deserialises the pickled Vocabulary object from disk.
vocab = Vocabulary.load('data/vocab.pkl')

# len(vocab) includes the two special tokens <PAD> (idx 0) and <UNK> (idx 1)
# plus up to 20,000 regular tokens, so the total is at most 20,002.
print(f'Vocabulary size: {len(vocab):,} tokens')

**Why we need the vocab size to instantiate the model:**

The first layer of `SentimentLSTM` is an `nn.Embedding` table of shape `(vocab_size, embedding_dim)`. Each row stores the learned dense vector for one token. PyTorch allocates this table at construction time, so `vocab_size` must be known before training begins.

Using a vocab size that is too small would cause an index-out-of-range error at runtime whenever the model encounters a token index ≥ `vocab_size`. Using the exact size from the saved vocabulary guarantees that every index produced by the tokenizer is valid.

## Section 2 — Define Model

We import `SentimentLSTM` from `src.model` and instantiate the baseline configuration:

| Hyperparameter | Value | Rationale |
|---|---|---|
| `vocab_size` | `len(vocab)` | Must match the tokenizer's vocabulary |
| `embedding_dim` | 128 | Standard starting point; balances capacity vs. training speed |
| `hidden_size` | 64 | Sufficient for binary classification on this dataset |
| `num_layers` | 1 | Baseline; a 2-layer variant is explored in Section 5 |
| `dropout` | 0.5 | Standard regularisation rate for LSTM classifiers |

In [ ]:
from src.model import SentimentLSTM

# Instantiate the baseline model.
# All hyperparameters match the design document's "Baseline" configuration.
model = SentimentLSTM(
    vocab_size=len(vocab),   # embedding table has one row per token
    embedding_dim=128,       # each token maps to a 128-dimensional vector
    hidden_size=64,          # LSTM maintains a 64-dimensional hidden state
    num_layers=1,            # single LSTM layer (baseline)
    dropout=0.5,             # 50% dropout on the final hidden state
)

print(model)

**Purpose of each architectural component:**

- **Embedding layer** (`nn.Embedding`): maps integer token indices to dense 128-dimensional vectors. Each token gets its own learnable vector that captures semantic meaning. `padding_idx=0` means the `<PAD>` token always produces a zero vector and receives no gradient updates — padding tokens should not influence the learned representations.

- **LSTM layer** (`nn.LSTM`): processes the sequence of embeddings left-to-right, maintaining a hidden state that accumulates contextual information from all tokens seen so far. At each time step the LSTM updates its hidden state based on the current embedding and the previous hidden state. We use the final hidden state (after the last token) as a fixed-size summary of the entire review. 64 hidden units is a good starting point for binary classification — large enough to capture sentiment patterns, small enough to train quickly.

- **Dropout layer** (`nn.Dropout`): randomly zeros 50% of the hidden-state activations during training. This forces the network to learn redundant representations and prevents it from relying too heavily on any single unit, which reduces overfitting. Dropout is automatically disabled during evaluation (`model.eval()`).

- **Linear + Sigmoid** (`nn.Linear` + `torch.sigmoid`): the linear layer projects the 64-dimensional hidden state down to a single scalar logit. The sigmoid activation squashes this logit into the range (0, 1), giving the probability that the review expresses positive sentiment. Values above 0.5 are classified as positive; values below 0.5 as negative.

## Section 3 — Model Summary

We print a summary of each named module along with its parameter count, then compute the total and trainable parameter counts. This gives us a clear picture of where the model's capacity is concentrated.

In [ ]:
# Print a dependency-free model summary: layer name and parameter count.
print(f"{'Layer':<30} {'Parameters':>15}")
print("-" * 47)

for name, module in model.named_modules():
    # named_modules() recurses into sub-modules; skip the top-level container
    if name == '':
        continue
    # Count parameters belonging directly to this module (not its children)
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        print(f"{name:<30} {params:>15,}")

print("-" * 47)

# Total and trainable parameter counts across the entire model
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"{'Total parameters':<30} {total_params:>15,}")
print(f"{'Trainable parameters':<30} {trainable_params:>15,}")

**Interpreting the parameter count:**

The parameter budget breaks down roughly as follows for the baseline model (vocab_size ≈ 20,002):

| Layer | Formula | Approximate count |
|---|---|---|
| Embedding | `vocab_size × embedding_dim` | 20,002 × 128 ≈ **2,560,256** |
| LSTM (1 layer) | `4 × (embedding_dim + hidden_size + 1) × hidden_size` | 4 × (128 + 64 + 1) × 64 ≈ **49,408** |
| Linear | `hidden_size × 1 + 1` (bias) | 64 + 1 = **65** |

The embedding layer dominates — it accounts for roughly **98%** of all parameters. This is typical for NLP models: the vocabulary lookup table is large, but the recurrent and classification layers are comparatively tiny. The practical implication is that reducing `embedding_dim` (e.g., from 128 to 64) would roughly halve the total parameter count, while doubling `hidden_size` would have a much smaller effect.

## Section 4 — Smoke Test

Before training, we verify that the model accepts the correct input shape and produces valid probability outputs. We create a dummy batch of 4 reviews, each represented as a sequence of 500 random token indices, and run a forward pass.

In [ ]:
# Set the model to evaluation mode so dropout is disabled during the smoke test.
# (We want deterministic output for verification, not stochastic dropout.)
model.eval()

# Create a dummy input: batch of 4 reviews, each with 500 token indices.
# torch.randint samples uniformly from [0, len(vocab)), matching the range
# of indices that the tokenizer would produce for real reviews.
x = torch.randint(0, len(vocab), (4, 500))

# Run the forward pass inside torch.no_grad() to skip gradient computation
# (not needed for inference; saves memory and speeds up the call).
with torch.no_grad():
    output = model(x)

# --- Assertions ---
# The output must have shape (batch_size, 1): one probability per review.
assert output.shape == (4, 1), f"Expected shape (4, 1), got {output.shape}"

# All values must be in [0, 1] because the final activation is sigmoid.
assert output.min().item() >= 0.0, f"Min value {output.min().item()} is below 0"
assert output.max().item() <= 1.0, f"Max value {output.max().item()} is above 1"

print("Smoke test passed!")
print(f"Output shape : {output.shape}")
print(f"Output values: {output.squeeze().tolist()}")
print(f"Min: {output.min().item():.6f}  Max: {output.max().item():.6f}")

**What the smoke test verifies:**

The smoke test checks three things before any training has occurred:

1. **Correct input shape handling**: the model accepts a `(batch_size, seq_len)` integer tensor without errors. This confirms that the embedding lookup, LSTM, and linear layers are wired together correctly.

2. **Correct output shape**: the output is `(batch_size, 1)` — one scalar per review. This is the shape expected by `BCELoss` during training.

3. **Valid probability range**: all output values are in `[0, 1]`, confirming that the sigmoid activation is applied correctly. A value close to 1 means the model (with random weights) happens to predict positive sentiment; close to 0 means negative. At this stage the values are essentially random — the model has not learned anything yet.

Running this test before training catches wiring bugs early, when they are easiest to fix.

## Section 5 — Two-Layer Variant (Experiment Preview)

In `04_training.ipynb` we will compare the baseline model against a two-layer variant with a larger hidden size. Here we preview that variant, run the same smoke test, and compare parameter counts.

**Hypothesis:** Adding a second LSTM layer and increasing hidden size should improve the model's ability to capture long-range dependencies, potentially improving validation accuracy by 1–2%. We will test this hypothesis in `04_training.ipynb`.

In [ ]:
# Instantiate the two-layer experimental variant.
# Differences from baseline: num_layers=2, hidden_size=128.
# When num_layers > 1, PyTorch automatically applies inter-layer dropout
# between the first and second LSTM layers at the specified rate.
model_2layer = SentimentLSTM(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_size=128,   # doubled from 64 to give the second layer more capacity
    num_layers=2,      # stack two LSTM layers
    dropout=0.5,
)

# --- Smoke test for the two-layer variant ---
model_2layer.eval()

# Reuse the same dummy input tensor from Section 4
with torch.no_grad():
    output_2layer = model_2layer(x)

# Verify the two-layer model produces the same output shape and valid probabilities
assert output_2layer.shape == (4, 1), f"Expected shape (4, 1), got {output_2layer.shape}"
assert output_2layer.min().item() >= 0.0
assert output_2layer.max().item() <= 1.0

print("Two-layer smoke test passed!")
print(f"Output shape : {output_2layer.shape}")
print(f"Output values: {output_2layer.squeeze().tolist()}")

# --- Parameter count comparison ---
baseline_params = sum(p.numel() for p in model.parameters())
twolayer_params = sum(p.numel() for p in model_2layer.parameters())

print()
print("Parameter count comparison:")
print(f"  Baseline  (1-layer, hidden=64) : {baseline_params:>10,}")
print(f"  Two-layer (2-layer, hidden=128): {twolayer_params:>10,}")
print(f"  Increase                       : {twolayer_params - baseline_params:>+10,} ({(twolayer_params/baseline_params - 1)*100:.1f}%)")

**Experiment hypothesis and rationale:**

The two-layer variant differs from the baseline in two ways:

1. **`num_layers=2`**: the first LSTM layer processes the embedded sequence and passes its full output sequence (all hidden states) to the second LSTM layer. The second layer then produces a higher-level representation. This is analogous to `return_sequences=True` in Keras — PyTorch's `nn.LSTM` handles the inter-layer connection automatically.

2. **`hidden_size=128`**: doubling the hidden size gives each LSTM layer more capacity to represent complex patterns. The trade-off is a larger model that takes longer to train and is more prone to overfitting.

**Hypothesis:** Adding a second LSTM layer and increasing hidden size should improve the model's ability to capture long-range dependencies, potentially improving validation accuracy by 1–2%. We will test this hypothesis in `04_training.ipynb`.

Note that the parameter increase is dominated by the LSTM layers (which now have more units and an extra layer) rather than the embedding table (which is unchanged). The embedding table still accounts for the majority of total parameters in both configurations.